In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [ ]:
silver_path     = "tihim_project.silver.customers"
gold_path       = "tihim_project.gold.dim_customers"
checkpoint_path = "/Volumes/tihim_project/ops/stream_state/checkpoints/gold/customers"

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_path} (
        customer_sk    STRING,
        customer_id    STRING,
        full_name      STRING,
        email          STRING,
        domain         STRING,
        city           STRING,
        state          STRING,
        is_active      BOOLEAN,
        start_date     TIMESTAMP,
        end_date       TIMESTAMP,
        create_date    TIMESTAMP,
        update_date    TIMESTAMP
    )
    USING DELTA
""")

In [ ]:
silver_stream = spark.readStream.option("readChangeFeed", "true").table(silver_path)

def prep_changes(df):
    df = df.filter(F.col("_change_type").isin("insert", "update_postimage"))
    w = Window.partitionBy("customer_id").orderBy(F.col("_commit_version").desc())
    return df.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn")

In [ ]:
def upsert_customers_to_gold(microbatch_df, batch_id):
    changes = prep_changes(microbatch_df)
    if changes.isEmpty():
        return

    dim = DeltaTable.forName(spark, gold_path)

    try:
        
        (dim.alias("target").merge( 
            source = changes.alias("update"),
            condition = """
                target.customer_id = update.customer_id AND 
                target.is_active = true
            """
        ).whenMatchedUpdate(
            condition = """
                NOT (target.city <=> update.city) OR 
                NOT (target.state <=> update.state)
            """,
            set = {
                "is_active": "false",
                 "end_date": "update.update_date"
                }
        
        ).execute())

        (dim.alias("target").merge(
            source = changes.alias("update"),
            condition = """
                target.customer_id = update.customer_id AND 
                target.is_active = true           
            """
        ).whenMatchedUpdate(
            
            condition = """
                NOT (target.email     <=> update.email)     OR
                NOT (target.full_name <=> update.full_name) OR
                NOT (target.domain    <=> update.domain)
            """,           
            set = {
                "email": "update.email",
                "full_name": "update.full_name",
                "domain": "update.domain",
                "update_date": "update.update_date"               
            }
        ).execute())

        
        new_versions = changes.alias("update").join(
            dim.toDF().alias("target"), 
            (F.col("update.customer_id") == F.col("target.customer_id")) & (F.col("target.is_active") == True),
            "left_anti")

        if not new_versions.isEmpty():
            (new_versions
                .select(
                    "customer_id",
                    "full_name", 
                    "email", 
                    "domain", 
                    "city", 
                    "state",
                    "create_date", 
                    "update_date")
                .withColumn("is_active",  F.lit(True))
                .withColumn("start_date", F.col("update_date"))
                .withColumn("end_date",   F.lit("2999-12-31").cast("timestamp"))
                .withColumn("customer_sk",
                    F.substring(F.sha2(F.concat_ws("||",
                        F.col("customer_id"), F.col("update_date").cast("string")), 256), 1, 16))
                .write.format("delta").mode("append").saveAsTable(gold_path))

    except Exception as e:
        print(f"[dim_customers] batch {batch_id} failed: {e}")
        raise

In [ ]:
query = (silver_stream.writeStream
    .foreachBatch(upsert_customers_to_gold)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()